# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [5]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [2]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [3]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [51]:
# ============================================================================
# Question 1
# ============================================================================
import pymongo

results = movies.find({"plot": {"$regex": "^war", "$options": "i"}}).sort('released', pymongo.ASCENDING).limit(5)
for i in results:
    print(f"Title: {i['title']} \nPlot: {i['plot']} \nRelease Date: {i['released']}\n")

Title: Nausicaè of the Valley of the Wind 
Plot: Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet. 
Release Date: 1984-03-11 00:00:00

Title: Nausicaè of the Valley of the Wind 
Plot: Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet. 
Release Date: 1984-03-11 00:00:00

Title: Heaven and Earth 
Plot: Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ... 
Release Date: 1991-02-08 00:00:00

Title: Under the Stars 
Plot: Warning! This synopsis contains spoilers Bajo las estrellas (beneath the stars) features the selfish... 
Release Date: 2007-06-15 00:00:00

Title: Aliens vs. Predator: Requiem 
Plot: Warring alien and predator races descend on a small town, where unsuspecting resid

In [50]:
# ============================================================================
# My study notes
# ============================================================================

# Duplicate movie record? 

for i in movies.find(
    {'title': {'$regex': "^Nausicaè of the Valley of the Wind"}}
):
    print(i)
    
# different last updated date & imdb votes, but same released date and plot. Could be a duplicate record?

{'_id': ObjectId('573a1398f29313caabce91ec'), 'plot': 'Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.', 'genres': ['Animation', 'Adventure', 'Fantasy'], 'runtime': 117, 'rated': 'PG', 'cast': ['Sumi Shimamoto', 'Mahito Tsujimura', 'Hisako Kyèda', 'Gorè Naya'], 'num_mflix_comments': 0, 'poster': 'https://m.media-amazon.com/images/M/MV5BODJiNmUzYmQtZTNhNS00NjY0LThmYjMtOTliOTM1NTdkYzY1XkEyXkFqcGdeQXVyNjU0OTQ0OTY@._V1_SY1000_SX677_AL_.jpg', 'title': 'Nausicaè of the Valley of the Wind', 'fullplot': 'In the far future, man has destroyed the Earth in the "Seven Days of Fire". Now, there are small pockets of humanity that survive. One pocket is the Valley of Wind where a princess named Nausicaè tries to understand, rather than destroy the Toxic Jungle. Note that the old US release titled Warriors of the Wind is an entirely kiddiefied version which edits the original movie heavily, thus creating an enti

Finding:

Documents in a MongoDB collection can store fields in different orders!

Documents in a MongoDB collection are not required to have the same number of fields!



### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [49]:
# ============================================================================
# Question 2
# ============================================================================
stage_group_rated = {
    "$group": {
        "_id": "$rated",
        "count": {"$sum": 1}
    }
}

pipeline = [stage_group_rated]

results = list(movies.aggregate(pipeline))

# Print the results
for i in results:
    print(f"{i['_id']} : {i['count']}") 

GP : 44
TV-PG : 76
None : 9894
TV-G : 59
G : 477
PG-13 : 2321
R : 5537
AO : 3
Not Rated : 1
PG : 1852
M : 37
TV-Y7 : 3
Approved : 5
PASSED : 181
TV-14 : 89
OPEN : 1
APPROVED : 709
TV-MA : 60



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [79]:
# ============================================================================
# Question 3
# ============================================================================
stage_match_with_3or_more_comments = {
   "$match": {
         "num_mflix_comments": {"$gte": 3}
   }
}

stage_count_movies = {
   "$count": "movies_with_3_or_more_comments"
}

pipeline = [
   stage_match_with_3or_more_comments,
   stage_count_movies,
]

results = movies.aggregate(pipeline)

# print(f"{list(results)[0]}")

print(f"Number of movies with 3 or more comments: {list(results)[0]['movies_with_3_or_more_comments']}")

Number of movies with 3 or more comments: 385
